# Three-Model Ablation Study

This notebook conducts a clean ablation study comparing:
- **Model A (Baseline)**: popularity, weight, market, year only
- **Model B (ENAO Similarity)**: baseline + ENAO co-occurrence weights
- **Model C (Hyperbolic Distance)**: baseline + hyperbolic distance

All models use identical Gradient Boosting hyperparameters.

## 1. Imports & Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

#random seed for reproducibility
np.random.seed(42)

In [2]:
DATASET = 'ablation_study.csv'
TARGET = 'log_streams'

df = pd.read_csv(DATASET)
print(f"✓ Dataset loaded: {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df = df.dropna(subset=['log_ranking_score_source', 'log_ranking_score_target'])

le_market = LabelEncoder()
df['market_enc'] = le_market.fit_transform(df['market'])

required_cols = ['distance', 'enao_similarity', 'log_ranking_score_source', 
                 'log_ranking_score_target', 'log_weight', 'market_enc', 'year']

missing = [col for col in required_cols if col not in df.columns]
if missing:
    print(f"ERROR: Missing columns: {missing}")
else:
    print(f"Dataset size: {len(df):,} collaborations")

✓ Dataset loaded: 43,661 rows
Columns: ['market', 'year', 'source', 'target', 'weight', 'avg_streams', 'distance', 'enao_similarity', 'popularity_source', 'popularity_target', 'ranking_source', 'ranking_target', 'ranking_score_source', 'ranking_score_target', 'log_streams', 'log_popularity_source', 'log_popularity_target', 'log_weight', 'log_ranking_score_source', 'log_ranking_score_target', 'distance_norm', 'distance_norm_global']
Dataset size: 43,661 collaborations


## 2. Define Feature Sets for Each Model

In [3]:
features_baseline = [
    'log_ranking_score_source',
    'log_ranking_score_target',
    'log_weight',
    'market_enc',
    'year'
]

features_A = features_baseline
features_B = features_baseline + ['enao_similarity']
features_C = features_baseline + ['distance']

features_all = features_baseline + ['distance', 'enao_similarity']


## 3. Train/Test Split

Use the same split for all three models (stratified by market)

In [4]:
y = df[TARGET]

X_full = df[features_all] 
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['market']
)

X_train_A = X_train_full[features_A]
X_test_A = X_test_full[features_A]

X_train_B = X_train_full[features_B]
X_test_B = X_test_full[features_B]

X_train_C = X_train_full[features_C]
X_test_C = X_test_full[features_C]

print(f"Train set: {len(X_train_A):,} samples")
print(f"Test set:  {len(X_test_A):,} samples")

Train set: 34,928 samples
Test set:  8,733 samples


## 4. Train All Three Models

Use identical hyperparameters (from the original prediction_model.ipynb that gave 0.684)

In [5]:
best_params = {
    'n_estimators': 300,
    'max_depth': 7,
    'learning_rate': 0.1,
    'subsample': 0.9,
    'min_samples_split': 5,
    'random_state': 42
}

print(f"Parameters: {best_params}")
print()

# ============================================================================
# Model A: Baseline
# ============================================================================

model_A = GradientBoostingRegressor(**best_params)
model_A.fit(X_train_A, y_train)

# 5-fold CV on training set
cv_scores_A = cross_val_score(model_A, X_train_A, y_train, cv=5, scoring='r2')
# Test set performance
r2_test_A = model_A.score(X_test_A, y_test)

print(f"  CV R²:   {cv_scores_A.mean():.3f} (±{cv_scores_A.std():.3f})")
print(f"  Test R²: {r2_test_A:.3f}")
print()

# ============================================================================
# Model B: ENAO Similarity
# ============================================================================

model_B = GradientBoostingRegressor(**best_params)
model_B.fit(X_train_B, y_train)

cv_scores_B = cross_val_score(model_B, X_train_B, y_train, cv=5, scoring='r2')
r2_test_B = model_B.score(X_test_B, y_test)

print(f"  CV R²:   {cv_scores_B.mean():.3f} (±{cv_scores_B.std():.3f})")
print(f"  Test R²: {r2_test_B:.3f}")
print()

# ============================================================================
# Model C: Hyperbolic Distance
# ============================================================================
model_C = GradientBoostingRegressor(**best_params)
model_C.fit(X_train_C, y_train)

cv_scores_C = cross_val_score(model_C, X_train_C, y_train, cv=5, scoring='r2')
r2_test_C = model_C.score(X_test_C, y_test)

print(f"  CV R²:   {cv_scores_C.mean():.3f} (±{cv_scores_C.std():.3f})")
print(f"  Test R²: {r2_test_C:.3f}")

Parameters: {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.1, 'subsample': 0.9, 'min_samples_split': 5, 'random_state': 42}

  CV R²:   0.658 (±0.002)
  Test R²: 0.665

  CV R²:   0.659 (±0.003)
  Test R²: 0.665

  CV R²:   0.677 (±0.002)
  Test R²: 0.685


## 5. Statistical Significance Testing

In [6]:
# Get predictions for all models on test set
pred_A = model_A.predict(X_test_A)
pred_B = model_B.predict(X_test_B)
pred_C = model_C.predict(X_test_C)

# Compute squared errors
errors_A = (y_test.values - pred_A) ** 2
errors_B = (y_test.values - pred_B) ** 2
errors_C = (y_test.values - pred_C) ** 2

# Paired t-tests
t_BA, p_BA = scipy_stats.ttest_rel(errors_B, errors_A)  # ENAO vs Baseline
t_CA, p_CA = scipy_stats.ttest_rel(errors_C, errors_A)  # Hyperbolic vs Baseline
t_CB, p_CB = scipy_stats.ttest_rel(errors_C, errors_B)  # Hyperbolic vs ENAO

print("="*80)
print("STATISTICAL SIGNIFICANCE TESTING")
print("="*80)
print()
print("Paired t-tests (comparing squared errors):")
print()
print(f"  B vs A (ENAO vs Baseline):")
print(f"    t = {t_BA:.4f}, p = {p_BA:.6f} {'✓ significant' if p_BA < 0.05 else '✗ not significant'}")
print()
print(f"  C vs A (Hyperbolic vs Baseline):")
print(f"    t = {t_CA:.4f}, p = {p_CA:.6f} {'✓ significant' if p_CA < 0.05 else '✗ not significant'}")
print()
print(f"  C vs B (Hyperbolic vs ENAO):")
print(f"    t = {t_CB:.4f}, p = {p_CB:.6f} {'✓ significant' if p_CB < 0.05 else '✗ not significant'}")
print()
print("="*80)

STATISTICAL SIGNIFICANCE TESTING

Paired t-tests (comparing squared errors):

  B vs A (ENAO vs Baseline):
    t = 0.2818, p = 0.778084 ✗ not significant

  C vs A (Hyperbolic vs Baseline):
    t = -7.3458, p = 0.000000 ✓ significant

  C vs B (Hyperbolic vs ENAO):
    t = -7.6048, p = 0.000000 ✓ significant

